# Walk-forward results

Per-fold predictive metrics from embargoed walk-forward validation. Load `data/results/xgb_wf.parquet` (and later LSTM) produced by `scripts/run_walkforward.py`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)

if Path.cwd().name == "notebooks":
    os.chdir("..")

xgb_path = Path("data/results/xgb_wf.parquet")
if not xgb_path.exists():
    raise FileNotFoundError(
        f"Missing {xgb_path}. Run: .venv/bin/python scripts/run_walkforward.py"
    )

preds = pd.read_parquet(xgb_path)
print(preds.head())
print(f"rows={len(preds):,} folds={sorted(preds['fold'].unique())}")

In [ ]:
def _signed_to_cls(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y).astype(np.int64)
    out = np.empty_like(y)
    out[y == -1] = 0
    out[y == 0] = 1
    out[y == 1] = 2
    return out


def fold_metrics(g: pd.DataFrame) -> dict:
    y_cls = _signed_to_cls(g["y_true"].to_numpy())
    probs = np.column_stack(
        [g["prob_down"].to_numpy(), g["prob_zero"].to_numpy(), g["prob_up"].to_numpy()]
    )
    y_pred = probs.argmax(axis=1)
    mask = y_cls != 1
    if mask.sum() >= 2 and len(np.unique(y_cls[mask])) == 2:
        auc = float(roc_auc_score((y_cls[mask] == 2).astype(int), probs[mask, 2]))
    else:
        auc = float("nan")
    return {
        "fold": int(g["fold"].iloc[0]),
        "accuracy": float(accuracy_score(y_cls, y_pred)),
        "log_loss": float(log_loss(y_cls, probs, labels=[0, 1, 2])),
        "macro_f1": float(
            f1_score(y_cls, y_pred, average="macro", labels=[0, 1, 2], zero_division=0)
        ),
        "auc_up_vs_down": auc,
        "n": len(g),
        "test_start": pd.Timestamp(g["ts_event"].min()),
    }


metrics = pd.DataFrame([fold_metrics(g) for _, g in preds.groupby("fold", sort=True)])
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
x = metrics["test_start"]

axes[0].plot(x, metrics["accuracy"], marker="o", label="XGBoost")
axes[0].axhline(metrics["accuracy"].mean(), color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("Per-fold accuracy")
axes[0].set_ylabel("accuracy")
axes[0].legend()

axes[1].plot(x, metrics["auc_up_vs_down"], marker="o", label="XGBoost")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="chance")
axes[1].set_title("Per-fold AUC (up vs down)")
axes[1].set_ylabel("AUC")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("test-day start")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Walk-forward XGBoost — stability across folds", y=1.02)
fig.tight_layout()
plt.show()

print(
    f"acc mean±std = {metrics['accuracy'].mean():.4f} ± {metrics['accuracy'].std():.4f}"
)
print(
    f"auc mean±std = {metrics['auc_up_vs_down'].mean():.4f} ± {metrics['auc_up_vs_down'].std():.4f}"
)